# LMA Phase 1 — Data Pipeline (Colab CPU)

Runs on **Colab CPU** (no GPU needed for Phase 1).
- **Data:** on Google Drive at `/content/drive/MyDrive/LMA_Project`
- **Code:** `lma_code.zip` — upload to Drive once, notebook unzips it
- **No GitHub, no auth, no giant cells**

**Steps:**
1. Upload `lma_code.zip` to your Drive `LMA_Project/` folder (one-time)
2. Mount Drive + unzip code + install deps
3. **Stream A:** Clean Nepali 22 GB → cleaned shards (~2–4 h)
4. **Stream B:** Download Hindi from HF + clean (parallel session)
5. Merge → dedup → splits
6. Train BPE tokenizers (minutes each)
7. Encode → measure tokens → stats
8. Tokenization examples

## 0. Mount Drive + unzip code + install deps

**Before running:** upload `lma_code.zip` to `MyDrive/LMA_Project/` in your Drive (drag-drop in browser).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJ = '/content/drive/MyDrive/LMA_Project'
import os; os.makedirs(PROJ, exist_ok=True)
print('Drive mounted at', PROJ)

In [ ]:
# Unzip the pipeline code (one-time)
import zipfile, os
zip_path = f'{PROJ}/lma_code.zip'
dest = '/content/lma_code'
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content/')
    print(f'Unzipped {len(os.listdir(dest))} dirs to {dest}')
else:
    print(f'ERROR: {zip_path} not found. Upload lma_code.zip to your Drive LMA_Project/ folder first!')

In [ ]:
import sys; sys.path.insert(0, '/content/lma_code')

# Quick import test
import common.clean, common.dedup, common.splits, common.stats, common.collect, common.langfilter
from hindi.tokenizer.tokenizer import Tokenizer as HiTok
from nepali.tokenizer.tokenizer import Tokenizer as NeTok
print('All imports OK — pipeline ready.')

In [ ]:
!pip install -q regex datasets
print('deps installed')

## Stream A — Nepali: clean 22 GB raw → cleaned shards

**Run in one Colab session.** Longest job (~2–4 h). Streams line-by-line, RAM stays bounded.

In [ ]:
# Clean Nepali 22 GB train file — streaming, single process
!cd /content/lma_code && python -m common.clean --lang nepali \
    --input /content/drive/MyDrive/LMA_Project/nepali/raw/downloaded/nepali_train.jsonl \
    --output-dir /content/drive/MyDrive/LMA_Project/nepali/processed/clean \
    --source-tag downloaded --shard-size 250000 --min-chars 80

In [ ]:
# Clean Nepali 5 GB test file
!cd /content/lma_code && python -m common.clean --lang nepali \
    --input /content/drive/MyDrive/LMA_Project/nepali/raw/downloaded/nepali_test.jsonl \
    --output-dir /content/drive/MyDrive/LMA_Project/nepali/processed/clean \
    --source-tag downloaded --shard-size 250000 --min-chars 80

In [ ]:
# Clean Nepali MANUAL shards (OCR + scraped)
import glob, subprocess, sys, os
manual_dirs = [
    '/content/drive/MyDrive/LMA_Project/nepali/raw/manual/extracted',
    '/content/drive/MyDrive/LMA_Project/nepali/raw/manual/sahityaras/extracted',
    '/content/drive/MyDrive/LMA_Project/nepali/raw/manual/cdc/extracted',
    '/content/drive/MyDrive/LMA_Project/nepali/raw/manual/national_library/ocr',
    '/content/drive/MyDrive/LMA_Project/nepali/raw/manual/national_library/extracted',
]
out_manual = '/content/drive/MyDrive/LMA_Project/nepali/processed/clean_manual'
for d in manual_dirs:
    for f in glob.glob(os.path.join(d, '*.jsonl')):
        print('cleaning', os.path.basename(f))
        subprocess.run([sys.executable, '-m', 'common.clean', '--lang', 'nepali',
                        '--input', f, '--output-dir', out_manual,
                        '--source-tag', 'manual', '--min-chars', '40'], cwd='/content/lma_code')
print('Nepali manual cleaning done')

## Stream B — Hindi: download 80% from HF + clean

**Run in a SECOND Colab session in parallel with Stream A.**

In [ ]:
# Download Hindi from HuggingFace (server-side streaming, ~4 GB target)
!cd /content/lma_code && python -m common.collect --lang hindi --sources wiki,mc4 \
    --output-dir /content/drive/MyDrive/LMA_Project/hindi/raw/downloaded \
    --max-chars 4000000000 --shard-size 250000

In [ ]:
# Clean Hindi downloaded shards
import glob, subprocess, sys, os
outd = '/content/drive/MyDrive/LMA_Project/hindi/processed/clean'
for f in sorted(glob.glob('/content/drive/MyDrive/LMA_Project/hindi/raw/downloaded/*.jsonl')):
    print('cleaning', os.path.basename(f))
    subprocess.run([sys.executable, '-m', 'common.clean', '--lang', 'hindi',
                    '--input', f, '--output-dir', outd,
                    '--source-tag', 'downloaded', '--shard-size', '250000',
                    '--min-chars', '80'], cwd='/content/lma_code')
print('Hindi downloaded cleaning done')

In [ ]:
# Clean Hindi MANUAL (webcrawl) shards
import glob, subprocess, sys, os
outm = '/content/drive/MyDrive/LMA_Project/hindi/processed/clean_manual'
for f in sorted(glob.glob('/content/drive/MyDrive/LMA_Project/hindi/raw/manual/**/*.jsonl', recursive=True)):
    print('cleaning manual', os.path.basename(f))
    subprocess.run([sys.executable, '-m', 'common.clean', '--lang', 'hindi',
                    '--input', f, '--output-dir', outm,
                    '--source-tag', 'manual', '--min-chars', '40'], cwd='/content/lma_code')
print('Hindi manual cleaning done')

## 4. Merge cleaned → near-dedup → splits

In [ ]:
# Combine downloaded + manual cleaned into one dir (symlinks)
import os, glob
for lang in ['hindi', 'nepali']:
    merged = f'/content/drive/MyDrive/LMA_Project/{lang}/processed/merged'
    os.makedirs(merged, exist_ok=True)
    for src_dir in [f'/content/drive/MyDrive/LMA_Project/{lang}/processed/clean',
                    f'/content/drive/MyDrive/LMA_Project/{lang}/processed/clean_manual']:
        if os.path.isdir(src_dir):
            for f in glob.glob(os.path.join(src_dir, '*.jsonl')):
                dst = os.path.join(merged, os.path.basename(f))
                if not os.path.exists(dst):
                    os.symlink(os.path.abspath(f), dst)
    print(lang, 'merged:', len(os.listdir(merged)), 'files')

In [ ]:
# Near-dedup (MinHash LSH)
for lang in ['hindi', 'nepali']:
    inp = f'/content/drive/MyDrive/LMA_Project/{lang}/processed/merged'
    out = f'/content/drive/MyDrive/LMA_Project/{lang}/processed/dedup'
    !cd /content/lma_code && python -m common.dedup --lang {lang} --input {inp} --output-dir {out} \
        --num-perm 64 --bands 8 --threshold 0.9 --min-chars 200

In [ ]:
# Splits (98/1/1, doc-level, seeded)
for lang in ['hindi', 'nepali']:
    inp = f'/content/drive/MyDrive/LMA_Project/{lang}/processed/dedup'
    out = f'/content/drive/MyDrive/LMA_Project/{lang}/splits'
    !cd /content/lma_code && python -m common.splits --lang {lang} --input {inp} \
        --output-dir {out} --train 0.98 --val 0.01 --test 0.01 --seed 17

## 5. Train tokenizers (BPE) — minutes each

In [ ]:
# Train Hindi BPE (subsample 80M chars, vocab 32000)
!cd /content/lma_code && python -m hindi.tokenizer.train --language hindi \
    --data /content/drive/MyDrive/LMA_Project/hindi/splits/hindi_train.jsonl \
    --vocab-size 32000 --char-budget 80000000 \
    --output /content/drive/MyDrive/LMA_Project/hindi/tokenizer/hindi_tokenizer.json

In [ ]:
# Train Nepali BPE (subsample 80M chars, vocab 32000)
!cd /content/lma_code && python -m nepali.tokenizer.train --language nepali \
    --data /content/drive/MyDrive/LMA_Project/nepali/splits/nepali_train.jsonl \
    --vocab-size 32000 --char-budget 80000000 \
    --output /content/drive/MyDrive/LMA_Project/nepali/tokenizer/nepali_tokenizer.json

## 6. Encode full corpora → measure tokens + stats

In [ ]:
# Corpus stats + token counts + manual/downloaded split + unknown rate
for lang in ['hindi', 'nepali']:
    splits = f'/content/drive/MyDrive/LMA_Project/{lang}/splits'
    tok = f'/content/drive/MyDrive/LMA_Project/{lang}/tokenizer/{lang}_tokenizer.json'
    heldout = f'{splits}/{lang}_heldout.jsonl'
    out = f'/content/drive/MyDrive/LMA_Project/{lang}/stats/{lang}_stats.json'
    !cd /content/lma_code && python -m common.stats --lang {lang} --input {splits} \
        --output {out} --tokenizer {tok} --heldout {heldout}

In [ ]:
# Display the stats
import json, os
for lang in ['hindi', 'nepali']:
    sp = f'/content/drive/MyDrive/LMA_Project/{lang}/stats/{lang}_stats.json'
    if os.path.exists(sp):
        print(f'\n===== {lang.upper()} STATS =====')
        s = json.load(open(sp))
        for k, v in s.items():
            if k != 'source_breakdown':
                print(f'  {k}: {v}')
        if 'source_breakdown' in s:
            print('  source_breakdown:')
            for kk, vv in s['source_breakdown'].items():
                print(f'    {kk}: {vv}')

## 7. Tokenization examples (for the report)

In [ ]:
import sys, os
sys.path.insert(0, '/content/lma_code')
from nepali.tokenizer.tokenizer import Tokenizer as NeTok
from hindi.tokenizer.tokenizer import Tokenizer as HiTok

examples = {
  'hindi': ['नमस्ते दुनिया, यह एक परीक्षण वाक्य है।',
            'भारत एक विशाल और विविध देश है।'],
  'nepali': ['नमस्ते संसार, यो एक परीक्षण वाक्य हो।',
             'नेपाल एक सुन्दर हिमालयको देश हो।'],
}
for lang, toks in [('hindi', HiTok), ('nepali', NeTok)]:
    tpath = f'/content/drive/MyDrive/LMA_Project/{lang}/tokenizer/{lang}_tokenizer.json'
    if not os.path.exists(tpath):
        print(f'{lang}: tokenizer not found'); continue
    tok = toks.load(tpath)
    print(f'\n===== {lang.upper()} tokenization examples =====')
    for ex in examples[lang]:
        ids = tok.encode(ex)
        print(f'text: {ex}')
        print(f'ids : {ids}')
        print(f'dec : {tok.decode(ids)}')
        print(f'round_trip: {tok.verify_round_trip(ex)}')
        print(f'stats: {tok.tokenization_stats(ex)}')
        print()

## 8. Done!

All artifacts on Drive:
- Cleaned: `LMA_Project/{lang}/processed/clean/`
- Splits: `LMA_Project/{lang}/splits/`
- Tokenizer: `LMA_Project/{lang}/tokenizer/{lang}_tokenizer.json`
- Stats: `LMA_Project/{lang}/stats/{lang}_stats.json`

Copy the stats + tokenizer JSONs to the repo on your Mac for git.